# Crisis Negotiator — GRPO Training (Kaggle 2×T4)

Trains **Qwen2.5-3B-Instruct** with GRPO on the Crisis Negotiator RL environment.

| | |
|---|---|
| **Hardware** | Kaggle 2×T4 (16 GB each) |
| **Precision** | fp16 (T4 does not support bf16) |
| **Runtime** | ~30–45 min for 64 prompts × 4 generations |
| **Output** | `crisis-negotiator-trained/` LoRA adapter + `crisis_training_log.json` + `reward_curve_training.png` |

**Settings → Accelerator → GPU T4 ×2** before running.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q "openenv-core>=0.2.3" "trl>=0.14" peft accelerate datasets sentence-transformers matplotlib
print('Dependencies installed.')

In [ ]:
# Cell 2 — Clone repo (kaggle_version branch)
import os, pathlib

if not pathlib.Path('server').exists():
    !git clone -b kaggle_version https://github.com/Dinesh052/Test.git .
    print('Cloned kaggle_version branch.')
else:
    print('Already inside repo.')

print('Files:', sorted(f for f in os.listdir('.') if not f.startswith('.'))[:14])

In [ ]:
# Cell 3 — Verify GPU setup
import torch
n = torch.cuda.device_count()
for i in range(n):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name} | {props.total_mem / 1024**3:.1f} GB | bf16={torch.cuda.is_bf16_supported()}')
print(f'\nTotal GPUs: {n}')
assert n >= 1, 'No GPU found! Enable GPU in Settings → Accelerator.'

In [ ]:
# Cell 4 — Smoke-test the environment
import sys
sys.path.insert(0, '.')

from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
print(f'Reset OK  | phase={obs.phase}  demands={[d["text"] for d in obs.stated_demands]}')

step_obs = env.step(NegotiatorAction(
    action_type='emotional_label',
    content="It sounds like you're feeling completely overwhelmed right now.",
    reasoning='open with empathy',
    target='hostage_taker',
))
print(f'Step OK   | reward={step_obs.reward:.4f}  phase={step_obs.phase}  done={step_obs.done}')
assert isinstance(step_obs.reward, float)
print('\nEnvironment smoke-test PASSED ✓')

In [ ]:
# Cell 5 — Patch train_local.py for Kaggle T4 GPUs
# T4 doesn't support bf16 → use fp16
# Reduce num_generations 8→4 for speed (still enough for GRPO advantage)
# Remove deepcopy in reward fn for ~2× speedup in scoring

import pathlib

src = pathlib.Path('train_local.py').read_text()

# 1. num_generations 8 → 4
src = src.replace(
    'num_generations: int = 8             # GRPO group size',
    'num_generations: int = 4             # GRPO group size (4 for Kaggle T4 speed)',
)

# 2. bf16=True → fp16=True in GRPOConfig
src = src.replace(
    'bf16=True,',
    'bf16=False,\n        fp16=True,',
)

# 3. torch_dtype bfloat16 → float16 in model loading
src = src.replace(
    'torch_dtype=torch.bfloat16,',
    'torch_dtype=torch.float16,',
)

# 4. Replace deepcopy env scoring with heuristic-only (major speedup)
# The deepcopy block starts at "env_copy = copy.deepcopy" and ends before "# Blend:"
src = src.replace(
    '''    env_step_reward = 0.0
    outcome_bonus = 0.0
    try:
        env_copy = copy.deepcopy(st["env"])
        stepped_obs = env_copy.step(action)
        env_step_reward = float(getattr(stepped_obs, "reward", 0.0))
        # Outcome bonus: large signal when episode terminates with surrender/release
        if getattr(stepped_obs, "done", False):
            msg = getattr(stepped_obs, "message", "")
            if any(kw in (msg or "").lower() for kw in [
                "voluntary_surrender", "hostage_released", "surrender", "released",
            ]):
                outcome_bonus = 0.40  # strong signal to learn surrender-inducing sequences
            elif any(kw in (msg or "").lower() for kw in [
                "harm_event", "tactical_intervention", "supervisor_termination",
            ]):
                outcome_bonus = -0.30  # avoid catastrophic outcomes
        bd["env_step"] = round(env_step_reward, 4)
        bd["outcome_bonus"] = round(outcome_bonus, 4)
        del env_copy
    except Exception as e:
        bd["env_step"] = 0.0
        bd["outcome_bonus"] = 0.0

    # Blend: 60% heuristic (shape), 40% real env reward (signal) + outcome bonus
    blended = 0.60 * score + 0.40 * env_step_reward + outcome_bonus
    blended = max(-0.5, min(1.0, blended))''',
    '''    # Kaggle: skip deepcopy env scoring for speed — use heuristic score directly
    bd["env_step"] = 0.0
    bd["outcome_bonus"] = 0.0
    blended = score  # pure heuristic — still rich signal from phase_align + demand_ack + diversity''',
)

pathlib.Path('train_local.py').write_text(src)
print('Patched train_local.py for Kaggle T4:')
print('  ✓ num_generations: 8 → 4')
print('  ✓ bf16 → fp16')
print('  ✓ deepcopy env scoring → heuristic-only (major speedup)')

In [ ]:
# Cell 6 — Run GRPO training
import os, subprocess, sys, time

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'

t0 = time.time()
result = subprocess.run(
    [sys.executable, 'train_local.py'],
    capture_output=False,
    text=True,
)
elapsed = (time.time() - t0) / 60
print(f'\nTraining finished in {elapsed:.1f} min. Exit code: {result.returncode}')

In [ ]:
# Cell 7 — Plot training reward curve
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open('crisis_training_log.json') as f:
    log = json.load(f)

steps   = [e['global_step'] for e in log]
rewards = [e['reward']      for e in log]
phases  = [e.get('obs_phase', 'opening') for e in log]

window  = min(16, len(rewards))
rolling = np.convolve(rewards, np.ones(window) / window, mode='valid')
roll_x  = steps[window - 1:]

PHASE_COLORS = {'opening': '#4C72B0', 'negotiation': '#DD8452', 'resolution': '#55A868'}
colours = [PHASE_COLORS.get(p, '#999999') for p in phases]

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(steps, rewards, c=colours, alpha=0.35, s=14, zorder=2)
ax.plot(roll_x, rolling, color='black', linewidth=2, label=f'rolling mean (w={window})', zorder=3)
ax.axhline(0.755, color='grey',    linestyle='--', linewidth=1, label='random baseline 0.755')
ax.axhline(0.950, color='#2ca02c', linestyle='--', linewidth=1, label='heuristic baseline 0.950')
ax.set_xlabel('GRPO Training Step')
ax.set_ylabel('Blended Episode Reward')
ax.set_title('Crisis Negotiator — GRPO Training Progress')

phase_patches = [mpatches.Patch(color=c, label=p) for p, c in PHASE_COLORS.items()]
line_handles, line_labels = ax.get_legend_handles_labels()
ax.legend(handles=line_handles + phase_patches,
          labels=line_labels + list(PHASE_COLORS.keys()),
          fontsize=7, loc='lower right', ncol=2)

plt.tight_layout()
plt.savefig('reward_curve_training.png', dpi=150)
plt.show()

print(f'Steps: {len(log)}')
print(f'Mean reward (all)    : {np.mean(rewards):.4f}')
print(f'Mean reward (last 10): {np.mean(rewards[-10:]):.4f}')
print(f'Improvement: {np.mean(rewards[-10:]) - np.mean(rewards[:10]):+.4f}')

In [ ]:
# Cell 8 — Eval: compare random / heuristic / trained
import subprocess, sys, json, pathlib

result = subprocess.run(
    [
        sys.executable, 'eval_baselines.py',
        '--n', '10',
        '--difficulties', 'easy,medium,hard',
        '--include-trained',
        '--adapter-dir', './crisis-negotiator-trained',
        '--base-model', 'Qwen/Qwen2.5-3B-Instruct',
    ],
    capture_output=False,
    text=True,
)
print('Exit code:', result.returncode)

if pathlib.Path('eval_summary.json').exists():
    summary = json.loads(pathlib.Path('eval_summary.json').read_text())
    print(f'\n{"Policy":15s}  {"Reward":>8s}  {"Surrender":>10s}  {"Steps":>6s}')
    print('-' * 46)
    for policy, s in summary.items():
        print(f"{policy:15s}  {s['mean_final_reward']:8.3f}  "
              f"{s['surrender_rate']:10.0%}  {s['mean_steps']:6.1f}")

In [ ]:
# Cell 9 — Save artifacts
import shutil, pathlib

artifacts = [
    'crisis-negotiator-trained',
    'crisis_training_log.json',
    'reward_curve_training.png',
    'eval_summary.json',
    'reward_curve.png',
]
for a in artifacts:
    p = pathlib.Path(a)
    if p.exists():
        print(f'  ✓ {a} ({p.stat().st_size / 1024:.0f} KB)' if p.is_file() else f'  ✓ {a}/ (dir)')
    else:
        print(f'  ✗ {a} — not found')

# Zip adapter for easy download
if pathlib.Path('crisis-negotiator-trained').exists():
    shutil.make_archive('crisis-negotiator-trained', 'zip', '.', 'crisis-negotiator-trained')
    print(f'\n📦 crisis-negotiator-trained.zip ready for download')